# 样本均值的标准化分布（正态与 t 分布）

> 对应代码：`MathStatsCode/code_in_notes/Chap.6/standard_normal_mean.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：分别用真实 σ 与样本标准差 s 标准化样本均值，对比其分布与标准正态、t(n-1) 密度的关系。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.6`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.6")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

本 notebook 对比样本均值的**两种标准化方式**。程序 `std_normal_mean` 返回两个统计量：`mean1=√n(x̄-μ)/σ`（用真实 σ 标准化）与 `mean2=√n(x̄-μ)/s`（用样本标准差 s 标准化，`r(sd)`）；随后用 `simulate` 重复20000次（每次样本量 `local N=8`，即自由度 df=7）得到两者的经验分布，为下一步与理论分布对比做准备。


In [ ]:
%%stata
clear
set more off
cap program drop std_normal_mean
program define std_normal_mean, rclass
	version 12
	syntax [,obs(integer 10) mu(real 0) sigma(real 1)]
	drop _all
	set obs `obs'
	tempvar x
	gen `x'=`sigma'*rnormal()+`mu'
	su `x'
	return scalar mean1=sqrt(`obs')*(r(mean)-`mu')/`sigma'
	return scalar mean2=sqrt(`obs')*(r(mean)-`mu')/r(sd)
end
local N=8
local df=`N'-1
simulate std_mean=r(mean1) sample_mean=r(mean2), reps(20000): std_normal_mean, obs(`N')


**standardised with sigma**

第一个统计量的理论分布：σ 已知时 √n(x̄-μ)/σ ~ N(0,1)，与样本量无关。这里对模拟结果排序，用 `normalden()` 计算标准正态密度曲线，与直方图叠加（`saving()` 先把图存成 .gph 文件供合并）。


In [ ]:
%%stata
sort std_mean
gen normal_den_std_mean=normalden(std_mean)
label variable normal_den_std_mean "Std Normal Density"
gen student_den_std_mean=tden(`df',std_mean)
label variable student_den_std_mean "t(`df') Density"
twoway (hist std_mean, density) (line normal_den_std_mean std_mean) (line student_den_std_mean std_mean) , graphr(fcolor(white) color(white))  xtitle("") saving(standard_normal_mean_1, replace)


**standardised with std deviation**

第二个统计量用 s 代替 σ 后不再服从正态，而是服从 **t 分布**（自由度 n-1=7）：理论上小样本时 t 分布比标准正态“尾部更厚”。同样叠加 `tden(7,·)` 的 t 密度曲线；最后 `graph combine` 把两张图并排合并、用 `graph export` 导出——对比可以看出：用 σ 标准化时正态曲线吻合，用 s 标准化时 t 曲线吻合。


In [ ]:
%%stata
sort sample_mean
gen normal_den_std_mean2=normalden(sample_mean)
label variable normal_den_std_mean2 "Std Normal Density"
gen t_den_std_sample_mean=tden(`df',sample_mean)
label variable t_den_std_sample_mean "t(`df') Density"
twoway (hist sample_mean, density) (line normal_den_std_mean2 sample_mean) (line t_den_std_sample_mean sample_mean), graphr(fcolor(white) color(white)) xtitle("") saving(standard_normal_mean_2, replace)

graph combine standard_normal_mean_1.gph standard_normal_mean_2.gph, graphr(fcolor(white) color(white)) col(2)
